In [16]:
import pandas as pd
import os
import yaml
import ast

# -------------------------
# PATH SETUP
# -------------------------

BASE_DIR = os.getcwd()              # current notebook folder (e.g. /notebooks)
PROJECT_DIR = os.path.dirname(BASE_DIR)   # go UP one level
OUTPUT_DIR = os.path.join(PROJECT_DIR, "sql_scripts")

# os.makedirs(OUTPUT_DIR, exist_ok=True)


# -------------------------
# LOAD DATA
# -------------------------
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)
df = pd.read_csv(config['output_data']['file'])
# -------------------------
# HELPERS
# -------------------------
# def split_list(x):
#     if pd.isna(x):
#         return []
#     return [i.strip() for i in str(x).split(",") if i.strip()]

def split_list(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except:
        return [i.strip(" []'\"") for i in str(x).split(",") if i.strip()]
# -------------------------
# CONTAINERS
# -------------------------
products = []
users = {}
reviews = []
product_images = []
categories =[]
product_categories =[]
category_lookup = {}
next_id = 1
image_id = 1
seen_reviews = set()
# -------------------------
# MAIN LOOP
# -------------------------
for _, row in df.iterrows():

    pid = row["product_id"]

    # PRODUCTS
    products.append({
        "product_id": pid,
        "product_name": row["product_name"],
        "about_product": row["about_product"].replace('',''),
        "discounted_price": row["discounted_price"],
        "actual_price": row["actual_price"],
        "discount_percentage": row["discount_percentage"],
        "rating": row["rating"],
        "rating_count": row["rating_count"],
        "product_link": row["product_link"]
    })

    # USERS + REVIEWS
    user_ids = split_list(row["user_id"])
    user_names = split_list(row["user_name"])

    review_ids = split_list(row["review_id"])
    titles = split_list(row["review_title"])
    contents = split_list(row["review_content"])

    # n = min(len(review_ids), len(titles), len(contents), len(user_ids))
    n = len(review_ids)
    for i in range(n):

        rid = review_ids[i]

        # SKIP duplicates
        if rid in seen_reviews:
            continue
        seen_reviews.add(rid)

        uid = user_ids[i] if i < len(user_ids) else None
        uname = user_names[i] if i < len(user_names) else None
        title = titles[i] if i < len(titles) else None
        content = contents[i] if i < len(contents) else None

        if uid and uid not in users:
            users[uid] = {
                "user_id": uid,
                "user_name": uname
            }

        reviews.append({
            "review_id": rid,
            "product_id": pid,
            "user_id": uid,
            "review_title": title,
            "review_content": content
        })

    # IMAGES
    imgs = split_list(row["img_link"])
    for img in imgs:
        product_images.append({
            "image_id": image_id,
            "product_id": pid,
            "image_url": img
        })
    image_id += 1

    category_name = row["category"]


    # Create category only once
    if category_name not in category_lookup:

        category_lookup[category_name] = next_id

        categories.append({
            "category_id": next_id,
            "category_name": category_name,

            "sub_cat1": row["category"].split("|")[0]
                            .strip()
                            .replace(",", "_")
                            .replace("&", "_"),

            "sub_cat2": row["category"].split("|")[1]
                            .strip()
                            .replace(",", "_")
                            .replace("&", "_"),

            "sub_cat3": row["category"].split("|")[-1]
                            .strip()
                            .replace(",", "_")
                            .replace("&", "_")
        })

        next_id += 1

    # Retrieve existing ID
    category_id = category_lookup[category_name]

    # Link product to category
    product_categories.append({
        "product_id": pid,
        "category_id": category_id
    })

# -------------------------
# EXPORT CSVs
# -------------------------
pd.DataFrame(products).to_csv(os.path.join(OUTPUT_DIR, "products.csv"), index=False)
pd.DataFrame(list(users.values())).to_csv(os.path.join(OUTPUT_DIR, "users.csv"), index=False)
pd.DataFrame(reviews).to_csv(os.path.join(OUTPUT_DIR, "reviews.csv"), index=False)
pd.DataFrame(product_images).to_csv(os.path.join(OUTPUT_DIR, "product_images.csv"), index=False)
pd.DataFrame(product_categories).to_csv(os.path.join(OUTPUT_DIR, "product_categories.csv"), index=False)
pd.DataFrame(categories).to_csv(os.path.join(OUTPUT_DIR, "categories.csv"), index=False)

# print(f"Done! Files saved to: {OUTPUT_DIR}")


In [14]:
with open(path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if line.count('"') % 2 != 0:
            print("Unbalanced quotes at line:", i)
            break
print('done')

done
